In [ ]:
# =====================================================================
# Side-by-Side Comparison: 4D Engine vs Dynamic Engine
#   Runs 3 test windows through both engines, prints outage difference.
# =====================================================================

import torch, numpy as np, math
from scipy.stats import gaussian_kde

try:
    device = torch.device('cuda'); torch.zeros(1, device=device)
except:
    device = torch.device('cpu')
print(f'Device: {device}')

# ============================================================
# Shared RWP Generator ((identical to both engines))
# ============================================================
def gen_rwp(rx,ry,sim_time,dt=10,nu=5,rng=None):
    if rng is None: rng=np.random
    rs=[0.0,rx,0.0,ry]; hc=np.array([rx/2,ry/2]); hr=min(rx,ry)/3.0
    p_t=0.6;p_s=0.3;tau_h=1.5;tau_n=0.1;v_h=0.2;v_n=1.0;g_h=0.6;g_n=0.2
    ts=int(sim_time/dt)
    def gt():
        t=hc+(rng.rand(2)-0.5)*2*hr if rng.rand()<g_h else np.array([rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])])
        t[0]=np.clip(t[0],rs[0],rs[1]);t[1]=np.clip(t[1],rs[2],rs[3]);return t
    uh=1.5+0.5*rng.rand(nu);pos=np.zeros((nu,ts,2))
    up=np.zeros((nu,2));ur=['normal']*nu;us=[None]*nu;ut_=np.zeros((nu,2));ud=np.zeros((nu,2));usp=np.zeros(nu);uprem=np.zeros(nu)
    for i in range(nu):
        up[i]=[rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])]
        d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
        if rng.rand()<p_t:us[i]='transfer';ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
        else:us[i]='pause';uprem[i]=rng.exponential(tau_h if ur[i]=='hot'else tau_n);pos[i,0,:]=up[i]
    for step in range(1,ts):
        for i in range(nu):
            if us[i]=='pause':
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:us[i]='transfer';ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
            else:
                md=usp[i]*dt;pd=ut_[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut_[i].copy();d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
                    if rng.rand()<p_s:us[i]='pause';uprem[i]=rng.exponential(tau_h if ur[i]=='hot'else tau_n)
                    else:ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
                else:up[i]=up[i]+ud[i]*md;pos[i,step,:]=up[i]
    pts=np.zeros((nu*ts,3));idx=0
    for u in range(nu):
        for s in range(ts):pts[idx]=[pos[u,s,0],pos[u,s,1],uh[u]];idx+=1
    return pts

# ============================================================
# Build BOTH physics engines for 10×10
# ============================================================
rx=10.0;ry=10.0;rz=3.0; xBs,yBs,zBs=10.0,-100.0,-10.0
E=8;dB_s=0.075;lam=0.075;Lp=2;dref=abs(yBs)*1.5
Pd=40.0;Rth=0.2;Nd=-174.0;Bw=20e6;pm_val=0.2;nu=5
PB=10**(Pd/10)*1e-3;N0=10**(Nd/10)*1e-3*Bw
Z_HS=[1.5,1.625,1.75,1.875,2.0];N_Z=5;GR=200

xv=torch.linspace(0,rx,GR,device=device);yv=torch.linspace(0,ry,GR,device=device)
Xg,Yg=torch.meshgrid(xv,yv,indexing='ij')
xyf=torch.stack([Xg.flatten(),Yg.flatten()],dim=1)
gl=[]
for zh in Z_HS:gl.append(torch.stack([Xg.flatten(),Yg.flatten(),torch.full_like(Xg.flatten(),zh)],dim=1))
gl=torch.cat(gl,dim=0);Ngrid=gl.shape[0]

# KDE from RWP (shared between both engines)
np.random.seed(777);et=gen_rwp(rx,ry,100000,10)
kde=gaussian_kde(et[:,:2].T,bw_method='scott')
wxy=kde(xyf.cpu().numpy().T).flatten();wxy=wxy/wxy.sum()
gw=torch.tensor(np.tile(wxy,N_Z),dtype=torch.float32,device=device);gw=gw/gw.sum()

# NLoS phases (shared)
np.random.seed(42)
ps=torch.tensor(2*np.pi*np.random.rand(Ngrid),dtype=torch.float32,device=device)
tt=torch.tensor(-np.pi+2*np.pi*np.random.rand(Ngrid),dtype=torch.float32,device=device)
eta=torch.tensor(10**((-15+5*np.random.rand(Ngrid))/10),dtype=torch.float32,device=device)
na=torch.arange(E,dtype=torch.float32,device=device)
dyB=torch.tensor(0.0-yBs,device=device)
v1c=lam/(4*math.pi);v1pc=-(2*math.pi/lam);oE=1/math.sqrt(E)

print(f'Grid: {Ngrid} points, gw sum={gw.sum().item():.4f}')

# ============================================================
# Engine 1: 4D-Style (batch channel processing)
# ============================================================
def phys_chan_4d(wp,lc):
    if not isinstance(wp,torch.Tensor):wp=torch.tensor(wp,dtype=torch.float32,device=device)
    if wp.dim()==1:wp=wp.unsqueeze(0)
    Bn=wp.shape[0];xc,zc,Lx,Lz=wp[:,0],wp[:,1],wp[:,2],wp[:,3]
    xu,yu,zu=lc[:,0],lc[:,1],lc[:,2]
    dx=xc-xBs;dy=dyB;dz=zc-zBs
    Rw=torch.sqrt(dx**2+dy**2+dz**2);th=torch.atan2(dy,dx);ph=torch.acos(dz/Rw)
    kx=torch.sin(ph)*torch.cos(th);kz=torch.cos(ph)
    x1=xc-Lx/2;z1=zc-Lz/2;x2=xc-Lx/2;z2=zc+Lz/2;x3=xc+Lx/2;z3=zc-Lz/2;x4=xc+Lx/2;z4=zc+Lz/2
    def rd(xs,zs):
        ddx=xs.unsqueeze(1)-xBs;ddy=dyB;ddz=zs.unsqueeze(1)-zBs
        L=torch.sqrt(ddx**2+ddy**2+ddz**2);return ddx/L,ddy/L,ddz/L
    ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
    ua=torch.stack([ux1,ux2,ux3,ux4],0);uz=torch.stack([uz1,uz2,uz3,uz4],0)
    umin=ua.min(0).values;umax=ua.max(0).values;zmin=uz.min(0).values;zmax=uz.max(0).values
    dux=xu-xBs;duy=yu-yBs;duz=zu-zBs;Lu=torch.sqrt(dux**2+duy**2+duz**2)
    uux=dux/Lu;uuz=duz/Lu
    ix=torch.sigmoid(1000*(uux-umin))*torch.sigmoid(1000*(umax-uux))
    iz=torch.sigmoid(1000*(uuz-zmin))*torch.sigmoid(1000*(zmax-uuz));il=ix*iz
    d2x=xu-xc.unsqueeze(1);d2y=yu;d2z=zu-zc.unsqueeze(1)
    Ru=torch.sqrt(d2x**2+d2y**2+d2z**2);t1=d2x/Ru;t2=d2z/Ru
    ax=(1-il)*(kx.unsqueeze(1)-t1);az=(1-il)*(kz.unsqueeze(1)-t2)
    sx=torch.sinc((math.pi/lam)*Lx.unsqueeze(1)*ax);sz=torch.sinc((math.pi/lam)*Lz.unsqueeze(1)*az)
    p1=(2*math.pi/lam)*dB_s*na.unsqueeze(0)*torch.sin(th).unsqueeze(1)
    a1=oE*torch.complex(torch.cos(p1),torch.sin(p1))
    v1m=v1c/Rw;v1p=v1pc*Rw;v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p))
    H1=v1.unsqueeze(1)*a1.conj()
    p2=(2*math.pi/lam)*dB_s*na.unsqueeze(0)*torch.sin(tt).unsqueeze(1)
    a2=oE*torch.complex(torch.cos(p2),torch.sin(p2))
    v2m_s=eta*(lam/(4*math.pi*dref));v2=torch.complex(v2m_s*torch.cos(ps),v2m_s*torch.sin(ps))
    H2=v2.unsqueeze(1)*a2.conj().unsqueeze(0)
    Ht=math.sqrt(E/Lp)*(H1.unsqueeze(1)+H2)
    fm=(Lx.unsqueeze(1)*Lz.unsqueeze(1)*sx*sz)/(lam*Ru)
    fp=(-(2*math.pi/lam)*(kx*xc+kz*zc)-(math.pi/2))
    fc=torch.complex(fm*torch.cos(fp.unsqueeze(1)),fm*torch.sin(fp.unsqueeze(1)))
    return Ht*fc.unsqueeze(2)

@torch.no_grad()
def outage_4d(X):
    out=[]
    for i in range(0,len(X),200):
        xb=X[i:i+200];th=torch.tensor(xb,dtype=torch.float32,device=device)
        He=phys_chan_4d(th,gl);Hw=torch.sum(He,dim=2)/math.sqrt(E)
        dp=(torch.abs(Hw)**2)*pm_val*PB;it=(nu-1)*dp;sr=dp/(it+N0)
        xc_,zc_=th[:,0],th[:,1];ab=math.pi/3.0;Ses=torch.zeros_like(sr)
        rn=torch.sqrt((gl[:,0].unsqueeze(0)-xc_.unsqueeze(1))**2+(gl[:,2].unsqueeze(0)-zc_.unsqueeze(1))**2+gl[:,1].unsqueeze(0)**2+1e-12)
        for a in [0.0,math.pi/2,math.pi,3*math.pi/2]:
            dot=torch.abs((gl[:,0].unsqueeze(0)-xc_.unsqueeze(1))*math.cos(a)+gl[:,1].unsqueeze(0)*math.sin(a))
            cp=dot/rn;ph_b=torch.acos(torch.clamp(cp,0,1))
            Se=(math.pi-torch.clamp(2*ab-ph_b,min=0))/math.pi;Ses=Ses+torch.clamp(Se,1/3,1)
        sr=((Ses/4)*dp)/((Ses/4)*it+N0)
        bits=(torch.log2(1+sr)<Rth).float()
        out.append(torch.sum(bits*gw,dim=1).cpu().numpy());torch.cuda.empty_cache()
    return np.concatenate(out)

# ============================================================
# Engine 2: Dynamic-Style (per-window channel)
# ============================================================
@torch.no_grad()
def outage_dyn(X):
    out=[]
    for i in range(0,len(X),200):
        b=torch.tensor(X[i:i+200],dtype=torch.float32,device=device);bo=torch.zeros(len(b),device=device)
        for j in range(len(b)):
            xc,zc,Lx,Lz=b[j,0],b[j,1],b[j,2],b[j,3]
            xg,yg,zg=gl[:,0],gl[:,1],gl[:,2];Ng=xg.shape[0]
            dx=xc-xBs;dy=dyB;dz=zc-zBs;Rw=torch.sqrt(dx**2+dy**2+dz**2);th=torch.atan2(dy,dx);ph=torch.acos(dz/Rw)
            kx=torch.sin(ph)*torch.cos(th);kz=torch.cos(ph)
            def rd(xs,zs):dd=xs-xBs;dz_=zs-zBs;L=torch.sqrt(dd**2+dy**2+dz_**2);return dd/L,dy/L,dz_/L
            x1,x2=xc-Lx/2,xc-Lx/2;x3,x4=xc+Lx/2,xc+Lx/2;z1,z3=zc-Lz/2,zc-Lz/2;z2,z4=zc+Lz/2,zc+Lz/2
            ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
            uma=torch.stack([ux1,ux2,ux3,ux4]);uzm=torch.stack([uz1,uz2,uz3,uz4])
            umin=uma.min(0).values;umax=uma.max(0).values;zmin=uzm.min(0).values;zmax=uzm.max(0).values
            dux=xg-xBs;duy=yg-yBs;duz=zg-zBs;Lu=torch.sqrt(dux**2+duy**2+duz**2);uux=dux/Lu;uuz=duz/Lu
            ix=torch.sigmoid(1000*(uux-umin))*torch.sigmoid(1000*(umax-uux))
            iz=torch.sigmoid(1000*(uuz-zmin))*torch.sigmoid(1000*(zmax-uuz));il=ix*iz
            d2x=xg-xc;d2y=yg;d2z=zg-zc;Ru=torch.sqrt(d2x**2+d2y**2+d2z**2);t1=d2x/Ru;t2=d2z/Ru
            ax=(1-il)*(kx-t1);az=(1-il)*(kz-t2)
            sx=torch.sinc((math.pi/lam)*Lx*ax);sz=torch.sinc((math.pi/lam)*Lz*az)
            p1=(2*math.pi/lam)*dB_s*na.unsqueeze(0)*torch.sin(th);a1=oE*torch.complex(torch.cos(p1),torch.sin(p1))
            v1m_s=v1c/Rw;v1p_s=v1pc*Rw;v1_s=torch.complex(v1m_s*torch.cos(v1p_s),v1m_s*torch.sin(v1p_s))
            H1_s=v1_s*a1.conj()
            p2=(2*math.pi/lam)*dB_s*na.unsqueeze(0)*torch.sin(tt).unsqueeze(1)
            a2_s=oE*torch.complex(torch.cos(p2),torch.sin(p2))
            v2m_s=eta*(lam/(4*math.pi*dref));v2_s=torch.complex(v2m_s*torch.cos(ps),v2m_s*torch.sin(ps))
            H2_s=v2_s.unsqueeze(1)*a2_s.conj()
            Ht_s=math.sqrt(E/Lp)*(H1_s+H2_s);fm_s=(Lx*Lz*sx*sz)/(lam*Ru)
            fp_s=(-(2*math.pi/lam)*(kx*xc+kz*zc)-(math.pi/2))
            fc_s=torch.complex(fm_s*torch.cos(fp_s),fm_s*torch.sin(fp_s))
            He_s=Ht_s*fc_s.unsqueeze(1);Hw_s=torch.sum(He_s,dim=1)/math.sqrt(E)
            dp_s=(torch.abs(Hw_s)**2)*pm_val*PB;it_s=(nu-1)*dp_s;sr_s=dp_s/(it_s+N0)
            ab=math.pi/3.0;Ses_s=torch.zeros(Ng,device=device)
            rn=torch.sqrt((xg-xc)**2+yg**2+(zg-zc)**2+1e-12)
            for a in [0.0,math.pi/2,math.pi,3*math.pi/2]:
                dot=torch.abs((xg-xc)*math.cos(a)+yg*math.sin(a));cp=dot/rn;ph_b=torch.acos(torch.clamp(cp,0,1))
                Se=(math.pi-torch.clamp(2*ab-ph_b,min=0))/math.pi;Ses_s=Ses_s+torch.clamp(Se,1/3,1)
            sr_se=((Ses_s/4)*dp_s)/((Ses_s/4)*it_s+N0);bits_s=(torch.log2(1+sr_se)<Rth).float()
            bo[j]=torch.sum(bits_s*gw)
        out.append(bo.cpu().numpy());torch.cuda.empty_cache()
    return np.concatenate(out)

# ============================================================
# Compare
# ============================================================
tests = np.array([
    [5.264,1.294,9.007,0.183],  # known knee
    [1.0,0.5,0.5,0.5],          # corner
    [5.0,1.5,10.0,0.3],         # full-width
])

print(f'\n{"Window":<40} {"4D":<10} {"Dynamic":<10} {"Diff":<10}')
print('-'*70)
for ti in range(len(tests)):
    t=tests[ti:ti+1]
    o4d = outage_4d(t)[0]*100
    odyn = outage_dyn(t)[0]*100
    desc = f'[{t[0,0]:.2f},{t[0,1]:.2f},{t[0,2]:.2f},{t[0,3]:.3f}]'
    print(f'{desc:<40} {o4d:<10.2f}% {odyn:<10.2f}% {abs(o4d-odyn):.2f}%')

# Also compare intermediate values for the knee point
print(f'\n--- Intermediate values for knee window ---')
# Just check gw stats
print(f'gw min={gw.min().item():.6f} max={gw.max().item():.6f} mean={gw.mean().item():.6f}')
print(f'gw[:10] = {gw[:10].cpu().numpy()}')
